## Camada Gold
<p>Na camada gold deste projeto, consumimos os dados gerados na camada Silver, gerando tabelas analíticas voltadas ao negócio</p>
Neste notebook, observamos
<ul>
<li>Agregações</li>
<li>Joins</li>
<li>Window functions</li>
<li>Consulta SQL</li>
<li>Pivotação</li>
</ul>

In [0]:
# Imports
from delta.tables import DeltaTable
from pyspark.sql import DataFrame
from pyspark.sql import functions as F
from pyspark.sql.window import Window

##### Definição de constantes

In [0]:
CATALOGO      = "ANP_Combustiveis"

silverSchema = "02_silver"
goldSchema   = "03_gold"

tablesSilver = {
    "PRECOS_REVENDA":   f"`{CATALOGO}`.`{silverSchema}`.`precos_revenda`",
    "VENDAS_MUNICIPIO": f"`{CATALOGO}`.`{silverSchema}`.`vendas_municipio`",
    "ESTADOS":          f"`{CATALOGO}`.`{silverSchema}`.`estados`"
}

tablesGold = {
    "RESUMO_PRECOS":         f"`{CATALOGO}`.`{goldSchema}`.`resumo_precos`",
    "MERCADO_MUNICIPAL":     f"`{CATALOGO}`.`{goldSchema}`.`mercado_municipal`",
    "INDICADORES_ESTADUAIS": f"`{CATALOGO}`.`{goldSchema}`.`indicadores_estaduais`"
}

fuelCategories = [
    "GASOLINA",
    "ETANOL",
    "DIESEL",
    "DIESEL S10",
    "GNV"
]

##### Definição da função auxiliar para atualizar tabela Dellta

In [0]:
# Função auxiliar para carregar dados na tabela Delta
def mergeToDelta(df: DataFrame, tableName: str, condition: str):
    deltaTable = DeltaTable.forName(spark, tableName)

    (
        deltaTable.alias("target")
        .merge(
            df.alias("source"),
            condition,
        )
        .whenNotMatchedInsertAll()
        .execute()
    )

    print(f"[Merge-DeltaTable] Carga concluída em: {tableName}")

### Resumo mensal dos preços
##### Leitura dos dados da camada Silver

In [0]:
# Leitura dos dados de preços da camada Silver
silverPricesDf = (
    spark.table(tablesSilver["PRECOS_REVENDA"])
    .select(
        "codigo_ibge",
        "municipio",
        "uf",
        "id_revenda",
        "produto",
        "familia_combustivel",
        "data_coleta",
        "valor_venda",
    )
)

# Leitura dos dados de vendas da camada Silver
silverSalesDf = (
    spark.table(tablesSilver["VENDAS_MUNICIPIO"])
    .select(
        "ano_referencia",
        "codigo_ibge",
        "municipio",
        "uf",
        "familia_combustivel",
        "volume_vendido",
    )
)

# Contagem total de registros
pricesTot = silverPricesDf.count()
salesTot  = silverSalesDf.count()

print('Total de registros disponiveis ns camadas Silver:')
print(f"Preços: {pricesTot:,}")
print(f"Verndas: {salesTot:,}")

##### Resumo mensal dos preços

In [0]:
# Leitura e agregação dos dados de preços por ano e município
pricesResumeDf = (
    silverPricesDf
    .withColumn("ano_referencia", F.year("data_coleta"))
    .withColumn("mes_referencia", F.month("data_coleta"))
    .groupBy(
        "ano_referencia",
        "mes_referencia",
        "codigo_ibge",
        "municipio",
        "uf",
        "produto",
        "familia_combustivel",
    )
    .agg(
        F.round(F.avg("valor_venda"), 3)
        .cast("decimal(10,3)")
        .alias("preco_medio"),

        F.expr("percentile_approx(valor_venda, 0.5, 10000)")
        .cast("decimal(10,3)")
        .alias("preco_mediano"),

        F.min("valor_venda")
        .cast("decimal(10,3)")
        .alias("preco_minimo"),

        F.max("valor_venda")
        .cast("decimal(10,3)")
        .alias("preco_maximo"),

        F.coalesce(
            F.stddev_samp("valor_venda"),
            F.lit(0.0),
        )
        .cast("double")
        .alias("desvio_padrao_preco"),

        (F.max("valor_venda") - F.min("valor_venda"))
        .cast("decimal(10,3)")
        .alias("amplitude_preco"),

        F.countDistinct("id_revenda")
        .cast("long")
        .alias("quantidade_postos"),

        F.count(F.lit(1))
        .cast("long")
        .alias("quantidade_observacoes"),
    )
    .withColumn("processado_gold_em", F.current_timestamp())
)

# Atualização da tabela Delta
mergeToDelta(
    pricesResumeDf,
    tablesGold["RESUMO_PRECOS"],
    """
        target.ano_referencia = source.ano_referencia AND
        target.mes_referencia = source.mes_referencia AND
        target.codigo_ibge = source.codigo_ibge AND
        target.produto = source.produto
    """,
)

# Preview
display(
    pricesResumeDf
    .orderBy(
        F.desc("ano_referencia"),
        F.desc("mes_referencia"),
        "uf",
        "municipio",
        "produto",
    )
    .limit(20)
)

### Contextos municipais
##### Agregação das tabelas
<p>Agregação das tabelas de preços e vendas por ano, codigo do municiio, estado e tipo de combustível</p>

In [0]:
# Agregação dos dados de preços por ano e município
anualPricesDf = (
    silverPricesDf
    .withColumn("ano_referencia", F.year("data_coleta"))
    .groupBy(
        "ano_referencia",
        "codigo_ibge",
        "municipio",
        "uf",
        "familia_combustivel",
    )
    .agg(
        F.round(F.avg("valor_venda"), 3)
        .cast("decimal(10,3)")
        .alias("preco_medio"),

        F.countDistinct("id_revenda")
        .cast("long")
        .alias("quantidade_postos"),
    )
)

# Agregação dos dados de vendas por ano e município
anualSalesDf = (
    silverSalesDf
    .groupBy(
        "ano_referencia",
        "codigo_ibge",
        "municipio",
        "uf",
        "familia_combustivel",
    )
    .agg(
        F.sum("volume_vendido")
        .cast("double")
        .alias("volume_vendido"),
    )
)

##### Avaliação do cenário anual dos combustiveis por estado
<p>Join das tabelas já agregadas e cálculo das variações anuais com window function (lag)</p>

In [0]:
# Agregação dos dados de mercado (preços + vendas) por ano e município
baseMarketDf = (
    anualPricesDf.alias("p")
    .join(
        anualSalesDf.alias("v"),
        (F.col("p.ano_referencia") == F.col("v.ano_referencia"))
        & (F.col("p.codigo_ibge") == F.col("v.codigo_ibge"))
        & (F.col("p.familia_combustivel") == F.col("v.familia_combustivel")),
        "inner",
    )
    .select(
        F.col("p.ano_referencia"),
        F.col("p.codigo_ibge"),
        F.col("p.municipio"),
        F.col("p.uf"),
        F.col("p.familia_combustivel"),
        F.col("p.preco_medio"),
        F.col("v.volume_vendido"),
        F.col("p.quantidade_postos"),
    )
)

# Janela para cálculo da variação anual de preço e vendas
window = (
    Window
    .partitionBy("codigo_ibge", "familia_combustivel")
    .orderBy("ano_referencia")
)

# Cálculo da variação anual de preço e vendas através de Window function (lag)
marketDf = (
    baseMarketDf
    .withColumn(
        "ano_anterior",
        F.lag("ano_referencia").over(window),
    )
    .withColumn(
        "preco_ano_anterior",
        F.lag("preco_medio").over(window),
    )
    .withColumn(
        "vendas_ano_anterior",
        F.lag("volume_vendido").over(window),
    )
)

# Cálculo da variação anual de mercado por município
municipalMarketDf = (
    marketDf
    .select(
        "ano_referencia",
        "codigo_ibge",
        "municipio",
        "uf",
        "familia_combustivel",
        "preco_medio",
        "volume_vendido",
        "quantidade_postos",

        F.when(
            F.col("preco_ano_anterior").isNull()
            | (F.col("ano_anterior") != F.col("ano_referencia") - 1)
            | (F.col("preco_ano_anterior") == 0),
            F.lit(None).cast("double"),
        )
        .otherwise(
            F.round(
                (
                    (F.col("preco_medio") - F.col("preco_ano_anterior"))
                    / F.col("preco_ano_anterior")
                ) * 100,
                2,
            ).cast("double")
        )
        .alias("variacao_anual_preco"),

        F.when(
            F.col("vendas_ano_anterior").isNull()
            | (F.col("ano_anterior") != F.col("ano_referencia") - 1)
            | (F.col("vendas_ano_anterior") == 0),
            F.lit(None).cast("double"),
        )
        .otherwise(
            F.round(
                (
                    (F.col("volume_vendido") - F.col("vendas_ano_anterior"))
                    / F.col("vendas_ano_anterior")
                ) * 100,
                2,
            ).cast("double")
        )
        .alias("variacao_anual_vendas"),

        F.current_timestamp().alias("processado_gold_em"),
    )
)

# Atualização da tabela Delta
mergeToDelta(
    municipalMarketDf,
    tablesGold["MERCADO_MUNICIPAL"],
    """
        target.ano_referencia = source.ano_referencia AND
        target.codigo_ibge = source.codigo_ibge AND
        target.familia_combustivel = source.familia_combustivel
    """,
)

# Preview
display(
    municipalMarketDf
    .orderBy(
        F.desc("ano_referencia"),
        "uf",
        "municipio",
        "familia_combustivel",
    )
    .limit(20)
)

##### Pivotação das categorias de combustiveis
<p>Com a pivotação identificamos de forma clara a variação dos preços médios dos combustíveis ao logo dos anos.</p>

##### Consulta com spark SQL
<p>Exibir maiores variações anuais dos preços.</p>

In [0]:
municipalMarketDf.createOrReplaceTempView("vw_mercado_municipal_gold")

# Consulta SQL  das maiores variações de preços por municipio
rankVariationsDf = spark.sql(
    """
        SELECT
            ano_referencia,
            municipio,
            uf,
            familia_combustivel,
            preco_medio,
            volume_vendido,
            variacao_anual_preco,
            variacao_anual_vendas
        FROM vw_mercado_municipal_gold
        WHERE variacao_anual_preco IS NOT NULL
        ORDER BY ABS(variacao_anual_preco) DESC
        LIMIT 20
    """
)

display(rankVariationsDf)

### Análise das métricas de negócio

In [0]:
# Definição dos dados de análise
year = 2024                    # Ano que se deseja analisar
fuel = ["GASOLINA", "ETANOL"]  # Combustíveis que se deseja analisar

In [0]:
## Preparação dos dados

# Seleciona dados da tabela silver para o ano que se deseja realizar a análise
stateSilverDf = (
    spark.table(tablesSilver["ESTADOS"])
    .filter((F.col("ano_populacao") == year))
    .select(
        "codigo_uf",
        "uf",
        "nome_uf",
        "ano_populacao",
        "populacao",
        "ano_area",
        "area_km2"
    )
)

# Cálculo do preço médio estadual
statesAvgPriceDf = (
    silverPricesDf
    .filter(
        (F.year("data_coleta") == year)
        & F.col("familia_combustivel").isin(fuel)
    )
    .withColumn("ano_referencia", F.year("data_coleta"))
    .groupBy("ano_referencia", "uf", "familia_combustivel")
    .agg(
        F.round(F.avg("valor_venda"), 3)
        .cast("decimal(10,3)")
        .alias("preco_medio")
    )
)

# Volume total vendido por estado
stateSalesDf = (
    silverSalesDf
    .filter(
        (F.col("ano_referencia") == year) &
        F.col("familia_combustivel").isin(fuel)
    )
    .groupBy("ano_referencia", "uf", "familia_combustivel",)
    .agg(
        F.sum("volume_vendido")
        .cast("double")
        .alias("volume_vendido")
    )
)

# Tabela unificada para [preço médio, volume vendido, população, area]
analysisBaseCalcDf = (
    statesAvgPriceDf
    .join(stateSalesDf, on=["ano_referencia", "uf", "familia_combustivel"], how="inner")
    .join(stateSilverDf, on="uf", how="inner")
    # Cálculo do volume médio vendido por habitante
    .withColumn(
        "litros_por_habitante",
        F.round(F.col("volume_vendido") / F.col("populacao"), 3)
    )
    # Cálculo do volume médio vendido por km²
    .withColumn(
        "litros_por_km2",
        F.round(F.col("volume_vendido") / F.col("area_km2"), 3)
    )
)

# Janelas dos dados de para métrica nas tabelas
priceWindow = (
    Window
    .partitionBy("ano_referencia", "familia_combustivel")
    .orderBy(F.desc("preco_medio"))
)

volumeWindow  = (
    Window
    .partitionBy("ano_referencia", "familia_combustivel")
    .orderBy(F.desc("volume_vendido"))
)

volumeByHabWindow = (
    Window
    .partitionBy("ano_referencia", "familia_combustivel")
    .orderBy(F.desc("litros_por_habitante"))
)

areaWindow = (
    Window
    .partitionBy("ano_referencia", "familia_combustivel")
    .orderBy(F.desc("litros_por_km2"))
)


# Ranking das métricas por Estado
stateMetricsDf = (
    analysisBaseCalcDf
    # Ranking por preço
    .withColumn("ranking_preco", F.dense_rank().over(priceWindow).cast("int"))
    # Ranking volume consumido
    .withColumn("ranking_consumo", F.dense_rank().over(volumeWindow).cast("int"))
    # Ranking volume consumido / hab
    .withColumn("ranking_consumo_habitante",F.dense_rank().over(volumeByHabWindow).cast("int"),)
    # Ranking volume consumido / area
    .withColumn("ranking_consumo_area", F.dense_rank().over(areaWindow).cast("int"))
    .select(
        "ano_referencia",
        "codigo_uf",
        "uf",
        "nome_uf",
        "familia_combustivel",
        "preco_medio",
        "volume_vendido",
        "ano_populacao",
        "populacao",
        "ano_area",
        "area_km2",
        "litros_por_habitante",
        "litros_por_km2",
        "ranking_preco",
        "ranking_consumo",
        "ranking_consumo_habitante",
        "ranking_consumo_area",

        F.current_timestamp().alias("processado_gold_em")
    )
)


# Atualizando dados na tabela Delta
mergeToDelta(stateMetricsDf, tablesGold["INDICADORES_ESTADUAIS"],
    """
        target.ano_referencia
            = source.ano_referencia
        AND target.codigo_uf
            = source.codigo_uf
        AND target.familia_combustivel
            = source.familia_combustivel
    """
)

<h6>Métrica 1 - Avaliação da variação anual dos preços de combustível</h6>

In [0]:
pricesPivotDf = (
    silverPricesDf
    .withColumn("ano_referencia", F.year("data_coleta"))
    .groupBy("ano_referencia")
    .pivot("familia_combustivel", fuelCategories)
    .agg(F.round(F.avg("valor_venda"), 3))
    .orderBy("ano_referencia")
)

display(pricesPivotDf)

<h6>Métrica 2.1 - Ranking de preço médio por estado</h6>

In [0]:
display(
    stateMetricsDf
    .select(
        "ano_referencia",
        "familia_combustivel",
        "uf",
        "nome_uf",
        "preco_medio"
    )
    .orderBy("familia_combustivel", "ranking_preco")
)

<h6>Métrica 2.2 - Ranking estadual de volume vendido</h6>

In [0]:
display(
    stateMetricsDf
    .select(
        "ano_referencia",
        "familia_combustivel",
        "ranking_consumo",
        "uf",
        "nome_uf",
        "volume_vendido"
    )
    .orderBy("familia_combustivel", "ranking_consumo")
)

<h6>Métrica 2.3 - Ranking estadual de média por habitante</h6>

In [0]:
display(
    stateMetricsDf
    .select(
        "ano_referencia",
        "familia_combustivel",
        "ranking_consumo_habitante",
        "uf",
        "nome_uf",
        "volume_vendido",
        "populacao",
        "litros_por_habitante"
    )
    .orderBy("familia_combustivel", "ranking_consumo_habitante")
)

<h6>Métrica 2.4 - Ranking estadual de [volume vendido / km²]</h6>

In [0]:
display(
    stateMetricsDf
    .select(
        "ano_referencia",
        "familia_combustivel",
        "ranking_consumo_area",
        "uf",
        "nome_uf",
        "volume_vendido",
        "area_km2",
        "litros_por_km2"
    )
    .orderBy("familia_combustivel", "ranking_consumo_area")
)